In [ ]:
import os
os.chdir('../..')
os.getcwd()
import autograd.numpy as anp
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ── 1. Boston Housing ────────────────────────────────────────────
# 506 samples, 13 features, target = median home value (MEDV)
# sklearn removed load_boston; use the original CSV from StatLib
from sklearn.datasets import fetch_openml
boston_raw = fetch_openml(name="boston", version=1, as_frame=True, parser="auto")
X_boston = boston_raw.data.values.astype(float)
y_boston = boston_raw.target.values.astype(float)

# ── 2. Naval Propulsion Plants ───────────────────────────────────
# 11,934 samples, 16 features, target = GT compressor decay coeff (col 17)
df_naval = pd.read_csv("benchmarks_august/datasets/naval_data.txt",
                        sep=r"\s+", header=None)
X_naval = df_naval.iloc[:, :16].values.astype(float)
y_naval = df_naval.iloc[:, 16].values.astype(float)
n_naval = 1000 
idx = np.random.RandomState(42).choice(len(df_naval), n_naval, replace=False)
X_naval, y_naval = X_naval[idx], y_naval[idx]

# ── 3. Energy Efficiency ─────────────────────────────────────────
# 768 samples, 8 features, target = Y1 (heating load)
df_energy = pd.read_excel("benchmarks_august/datasets/energy_data.xlsx")
X_energy = df_energy.iloc[:, :8].values.astype(float)
y_energy = df_energy.iloc[:, 8].values.astype(float)

# ── Standardize & split (Hernández-Lobato & Adams convention) ────
datasets = {}
for name, X, y in [("boston", X_boston, y_boston),
                    ("naval",  X_naval,  y_naval),
                    ("energy", X_energy, y_energy)
                    ]:
    # 90/10 train-test split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    # Standardize using training statistics
    x_mean, x_std = X_tr.mean(axis=0), X_tr.std(axis=0)
    x_std[x_std == 0] = 1.0   # guard against constant columns (Naval has some)
    y_mean, y_std = y_tr.mean(), y_tr.std()

    X_tr = (X_tr - x_mean) / x_std
    X_te = (X_te - x_mean) / x_std
    y_tr = (y_tr - y_mean) / y_std
    y_te = (y_te - y_mean) / y_std

    datasets[name] = {
        "X_train": X_tr, "y_train": y_tr,
        "X_test":  X_te, "y_test":  y_te,
        "x_mean": x_mean, "x_std": x_std,
        "y_mean": y_mean, "y_std": y_std,
    }
    print(f"{name:8s}  N={X.shape[0]:>6d}  D={X.shape[1]:>2d}  "
          f"train={X_tr.shape[0]}  test={X_te.shape[0]}")

## BOSTON

In [ ]:
from benchmarks_august.targets.bnn_regression import bnn_regression
from benchmarks_august.samplers.warmstart.bnn_regression_warmstart import adam_fisher_regression
from benchmarks_august.samplers import build_sampler, apply_preprocess

H1=20
H2=10
H3=4

layers = [datasets['boston']['X_train'].shape[1], H1, H2, 1]

# 1. Build target (E and gradE come from here)
target_boston = bnn_regression(datasets['boston']['X_train'], datasets['boston']['y_train'], layer_sizes=layers,
    prior={"kind": "layered_gaussian",
           "sigma_w_layers": [1.0, 1.0, 1.0],
           "sigma_b_layers": [1.0, 1.0, 1.0]})

# 2. Warmstart (sets x_ref and Sigma_inv)
ws = adam_fisher_regression(target_boston, n_epochs=3000, lr=3e-3)
target_boston.x_ref = ws["x_ref"]
target_boston.Sigma_inv = ws["Sigma_inv"]

# 3. Build kappa from target.meta
w_prior = 0.5
kappa = np.empty(target_boston.D)
kappa[~target_boston.meta["weight_mask"]] = 1e6
for l, (w_sl, b_sl) in enumerate(target_boston.meta["slices"]):
    sigma_l = target_boston.meta["sigma_w_layers"][l]
    kappa[w_sl] = (1 - w_prior) / w_prior / (sigma_l * np.sqrt(2 * np.pi))

In [ ]:
N=10000
sampler_raw_boston = build_sampler("boomerang_pli", target_boston, N= N,
                        refresh_rate=1.0)
apply_preprocess(sampler_raw_boston, target_boston, {"method": "manual"})
print("-------------Boomerang-------------\n")
sampler_raw_boston.sample_auto()

sampler_sticky_raw_boston = build_sampler("sticky_boomerang_pli", target_boston, N=N,
                        refresh_rate=1.0, kappa=kappa)
apply_preprocess(sampler_sticky_raw_boston, target_boston, {"method": "manual"})
print("-------------Sticky Boomerang-------------\n")
sampler_sticky_raw_boston.sample_auto()

In [ ]:
from benchmarks_august.samplers.warmstart import warmup_reference

sampler_boston = build_sampler("boomerang_pli", target_boston, N=N,
                        refresh_rate=1.0)
print("-------------Boomerang adapting...-------------\n")
warmup_reference(sampler_boston, n_rounds=3, n_pilot=1000, tune_refresh=True)
sampler_boston.reset(N=N)
print("-------------Boomerang adapted-------------\n")
sampler_boston.sample_auto()

sampler_sticky_boston = build_sampler("sticky_boomerang_pli", target_boston, N=N,
                        refresh_rate=1.0, kappa=kappa, cold_start_threshold=0.05)
print("-------------Sticky Boomerang adapting...-------------\n")
warmup_reference(sampler_sticky_boston, n_rounds=3, n_pilot=1000, tune_refresh=True)
sampler_sticky_boston.reset(N=N)
print("-------------Sticky Boomerang adapted-------------\n")
sampler_sticky_boston.sample_auto()

In [ ]:
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path
t_raw_boston, x_raw_boston = resample_pdmp_path(sampler_raw_boston, n_samples=50000)
t_boston, x_boston = resample_pdmp_path(sampler_boston, n_samples=50000)

from sazz.samplers.boomerang_sampler.utils import resample_sticky_pdmp_path
t_sticky_raw_boston, x_sticky_raw_boston = resample_sticky_pdmp_path(sampler_sticky_raw_boston, n_samples=50000)
t_sticky_boston, x_sticky_boston = resample_sticky_pdmp_path(sampler_sticky_boston, n_samples=50000)

In [ ]:
from benchmarks_august.analysis.metrics import bnn_regression_performance

bnn_regression_performance(datasets['boston']['X_train'], 
                           datasets['boston']['y_train'], 
                           datasets['boston']['X_test'], 
                           datasets['boston']['y_test'],
                           datasets['boston']['y_std'],
    {
        "Adam MAP": target_boston.x_ref.reshape(1, -1),
        "Boomerang raw": x_raw_boston,
        "Sticky raw": x_sticky_raw_boston,
        "Boomerang": x_boston,
        "Sticky": x_sticky_boston,
    },
    shapes=target_boston.meta["shapes"],
    slices=target_boston.meta["slices"],
    weight_mask=target_boston.meta["weight_mask"])

## NAVAL

In [ ]:
layers = [datasets['naval']['X_train'].shape[1], H1, H2, 1]

# 1. Build target (E and gradE come from here)
target_naval = bnn_regression(datasets['naval']['X_train'], datasets['naval']['y_train'], layer_sizes=layers,
    prior={"kind": "layered_gaussian",
           "sigma_w_layers": [1.0, 1.0, 1.0],
           "sigma_b_layers": [1.0, 1.0, 1.0]})

# 2. Warmstart (sets x_ref and Sigma_inv)
ws = adam_fisher_regression(target_naval, n_epochs=3000, lr=3e-3)
target_naval.x_ref = ws["x_ref"]
target_naval.Sigma_inv = ws["Sigma_inv"]

# 3. Build kappa from target.meta
w_prior = 0.5
kappa = np.empty(target_naval.D)
kappa[~target_naval.meta["weight_mask"]] = 1e6
for l, (w_sl, b_sl) in enumerate(target_naval.meta["slices"]):
    sigma_l = target_naval.meta["sigma_w_layers"][l]
    kappa[w_sl] = (1 - w_prior) / w_prior / (sigma_l * np.sqrt(2 * np.pi))

In [ ]:
N=10000
sampler_raw_naval = build_sampler("boomerang_pli", target_naval, N= N,
                        refresh_rate=1.0)
apply_preprocess(sampler_raw_naval, target_naval, {"method": "manual"})
print("-------------Boomerang-------------\n")
sampler_raw_naval.sample_auto()

sampler_sticky_raw_naval = build_sampler("sticky_boomerang_pli", target_naval, N=N,
                        refresh_rate=1.0, kappa=kappa)
apply_preprocess(sampler_sticky_raw_naval, target_naval, {"method": "manual"})
print("-------------Sticky Boomerang-------------\n")
sampler_sticky_raw_naval.sample_auto()

In [ ]:
from benchmarks_august.samplers.warmstart import warmup_reference

sampler_naval = build_sampler("boomerang_pli", target_naval, N=N,
                        refresh_rate=1.0)
print("-------------Boomerang adapting...-------------\n")
warmup_reference(sampler_naval, n_rounds=3, n_pilot=1000, tune_refresh=True)
sampler_naval.reset(N=N)
print("-------------Boomerang adapted-------------\n")
sampler_naval.sample_auto()

sampler_sticky_naval = build_sampler("sticky_boomerang_pli", target_naval, N=N,
                        refresh_rate=1.0, kappa=kappa, cold_start_threshold=0.05)
print("-------------Sticky Boomerang adapting...-------------\n")
warmup_reference(sampler_sticky_naval, n_rounds=3, n_pilot=1000, tune_refresh=True)
sampler_sticky_naval.reset(N=N)
print("-------------Sticky Boomerang adapted-------------\n")
sampler_sticky_naval.sample_auto()

In [ ]:
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path
t_raw_naval, x_raw_naval = resample_pdmp_path(sampler_raw_naval, n_samples=50000)
t_naval, x_naval = resample_pdmp_path(sampler_naval, n_samples=50000)

from sazz.samplers.boomerang_sampler.utils import resample_sticky_pdmp_path
t_sticky_raw_naval, x_sticky_raw_naval = resample_sticky_pdmp_path(sampler_sticky_raw_naval, n_samples=50000)
t_sticky_naval, x_sticky_naval = resample_sticky_pdmp_path(sampler_sticky_naval, n_samples=50000)

In [ ]:
from benchmarks_august.analysis.metrics import bnn_regression_performance

bnn_regression_performance(datasets['naval']['X_train'], 
                           datasets['naval']['y_train'], 
                           datasets['naval']['X_test'], 
                           datasets['naval']['y_test'],
                           datasets['naval']['y_std'],
    {
        "Adam MAP": target_naval.x_ref.reshape(1, -1),
        "Boomerang raw": x_raw_naval,
        "Sticky raw": x_sticky_raw_naval,
        "Boomerang": x_naval,
        "Sticky": x_sticky_naval,
    },
    shapes=target_naval.meta["shapes"],
    slices=target_naval.meta["slices"],
    weight_mask=target_naval.meta["weight_mask"])

## ENERGY

In [ ]:
layers = [datasets['energy']['X_train'].shape[1], H1, H2, 1]

# 1. Build target (E and gradE come from here)
target_energy = bnn_regression(datasets['energy']['X_train'], datasets['energy']['y_train'], layer_sizes=layers,
    prior={"kind": "layered_gaussian",
           "sigma_w_layers": [1.0, 1.0, 1.0],
           "sigma_b_layers": [1.0, 1.0, 1.0]})

# 2. Warmstart (sets x_ref and Sigma_inv)
ws = adam_fisher_regression(target_energy, n_epochs=3000, lr=3e-3)
target_energy.x_ref = ws["x_ref"]
target_energy.Sigma_inv = ws["Sigma_inv"]

# 3. Build kappa from target.meta
w_prior = 0.5
kappa = np.empty(target_energy.D)
kappa[~target_energy.meta["weight_mask"]] = 1e6
for l, (w_sl, b_sl) in enumerate(target_energy.meta["slices"]):
    sigma_l = target_energy.meta["sigma_w_layers"][l]
    kappa[w_sl] = (1 - w_prior) / w_prior / (sigma_l * np.sqrt(2 * np.pi))

In [ ]:
N=10000
sampler_raw_energy = build_sampler("boomerang_pli", target_energy, N= N,
                        refresh_rate=1.0)
apply_preprocess(sampler_raw_energy, target_energy, {"method": "manual"})
print("-------------Boomerang-------------\n")
sampler_raw_energy.sample_auto()

sampler_sticky_raw_energy = build_sampler("sticky_boomerang_pli", target_energy, N=N,
                        refresh_rate=1.0, kappa=kappa)
apply_preprocess(sampler_sticky_raw_energy, target_energy, {"method": "manual"})
print("-------------Sticky Boomerang-------------\n")
sampler_sticky_raw_energy.sample_auto()

In [ ]:
from benchmarks_august.samplers.warmstart import warmup_reference

sampler_energy = build_sampler("boomerang_pli", target_energy, N=N,
                        refresh_rate=1.0)
print("-------------Boomerang adapting...-------------\n")
warmup_reference(sampler_energy, n_rounds=3, n_pilot=1000, tune_refresh=True)
sampler_energy.reset(N=N)
print("-------------Boomerang adapted-------------\n")
sampler_energy.sample_auto()

sampler_sticky_energy = build_sampler("sticky_boomerang_pli", target_energy, N=N,
                        refresh_rate=1.0, kappa=kappa, cold_start_threshold=0.05)
print("-------------Sticky Boomerang adapting...-------------\n")
warmup_reference(sampler_sticky_energy, n_rounds=3, n_pilot=1000, tune_refresh=True)
sampler_sticky_energy.reset(N=N)
print("-------------Sticky Boomerang adapted-------------\n")
sampler_sticky_energy.sample_auto()

In [ ]:
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path
t_raw_energy, x_raw_energy = resample_pdmp_path(sampler_raw_energy, n_samples=50000)
t_energy, x_energy = resample_pdmp_path(sampler_energy, n_samples=50000)

from sazz.samplers.boomerang_sampler.utils import resample_sticky_pdmp_path
t_sticky_raw_energy, x_sticky_raw_energy = resample_sticky_pdmp_path(sampler_sticky_raw_energy, n_samples=50000)
t_sticky_energy, x_sticky_energy = resample_sticky_pdmp_path(sampler_sticky_energy, n_samples=50000)

In [ ]:
from benchmarks_august.analysis.metrics import bnn_regression_performance

bnn_regression_performance(datasets['energy']['X_train'], 
                           datasets['energy']['y_train'], 
                           datasets['energy']['X_test'], 
                           datasets['energy']['y_test'],
                           datasets['energy']['y_std'],
    {
        "Adam MAP": target_energy.x_ref.reshape(1, -1),
        "Boomerang raw": x_raw_energy,
        "Sticky raw": x_sticky_raw_energy,
        "Boomerang": x_energy,
        "Sticky": x_sticky_energy,
    },
    shapes=target_energy.meta["shapes"],
    slices=target_energy.meta["slices"],
    weight_mask=target_energy.meta["weight_mask"])